In [5]:
import joblib
import numpy as np
import pandas as pd

def predict_match_winner(batting_team, bowling_team, venue, cum_runs, cum_wickets, overs_completed, target):
    # Load the trained model and encoders
    model = joblib.load('final_rf_model.pkl')
    encoders = joblib.load('encoders.pkl')

    # Encode categorical features
    try:
        batting_team_encoded = encoders['batting_team'].transform([batting_team])[0]
        bowling_team_encoded = encoders['bowling_team'].transform([bowling_team])[0]
        venue_encoded = encoders['venue_canonical'].transform([venue])[0]
    except ValueError as e:
        print("Error: One or more input values are not in the trained encoder's classes.")
        return

    # Compute additional features
    current_run_rate = cum_runs / overs_completed if overs_completed > 0 else 0
    remaining_overs = 20 - overs_completed
    required_run_rate = (target - cum_runs) / remaining_overs if remaining_overs > 0 else 0

    # Create input array for prediction
    input_features = np.array([[
        2,  # Inning is always 2 for prediction
        cum_runs,
        cum_wickets,
        current_run_rate,
        required_run_rate,
        target,
        batting_team_encoded,
        bowling_team_encoded,
        venue_encoded
    ]])

    # Make prediction
    prediction = model.predict(input_features)[0]
    predicted_probabilities = model.predict_proba(input_features)[0]
    result = "Win" if prediction == 1 else "Lose"

    return result, predicted_probabilities

# Example user input
batting_team = "Royal Challengers Bengaluru"
bowling_team = "Chennai Super Kings"
venue = "M Chinnaswamy Stadium"
cum_runs = 85
cum_wickets = 2
overs_completed = 10
target = 180

# Make prediction
match_result,predicted_probabilities = predict_match_winner(batting_team, bowling_team, venue, cum_runs, cum_wickets, overs_completed, target)
print(f"Predicted Outcome: {batting_team} {match_result}")
print("Prediction Probabilities (Loss, Win):", predicted_probabilities)


Predicted Outcome: Royal Challengers Bengaluru Lose
Prediction Probabilities (Loss, Win): [0.57043084 0.42956916]


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
